# Pairwise sequence identity estimation for the candidate independence test: are 12 candidates 12 designs?

#### Goal
Estimate, whether 12 candidates are genuinely 12 different designs, or just a few designs with minor variantion between them.

#### Assumption
Every design contains the same frozen binding motif: the same ~36 residues, the same amino acids, at the same structural positions. Any two designs therefore share those positions automatically — not because sequence design converged on them, but because the method put them there. Therefore, a pairwise identity of 54% is not 54% worth of convergence. Part of it is the motif, built in by construction.




#### Methodology
1. Sequence: All 66 pairs aligned globally using Needleman-Wunsch, following BLOSUM62 substitution matrix, gap open -11, gap extend -1. Identity reported as matched nongap residues divided by the length ofthe shorter sequence.
2. Structure: All-vs-all TM-score via TM-align on Cα coordinates of the ESMFold models. Each pair reported as the larger of the two chain normalisations, i.e. normalised by the shorter chain.
3. Relationship between 2: Pearson correlation between the two matrices over the 66 unique off-diagonal pairs.
4. External scale. TM-score of each candidate against Neo-2/15, pdb:6DG6, chain A and native IL-2, pdb:2B5I, chain A, with the Neo-2/15:IL-2 TM-score as a reference point for what an unrelated fold pair scores.

#### Limitations
TM-score and sequence identity both measure similarity, not independence in any causal sense. Twelve designs from one RFdiffusion checkpoint, one motif and two length budgets are correlated by construction; this notebook quantifies how much, and cannot establish what the accessible design space actually contains.

In [ ]:
# Setup and the TM-align helper. tmscore returns the larger of the two
# chain normalisations, i.e. normalised by the shorter chain, which is
# the generous direction for length-mismatched pairs: a deliberate
# choice so that similarity is not understated
import os, json, itertools
import numpy as np, pandas as pd
import subprocess

from google.colab import drive
drive.mount("/content/drive")

PROJ = "/content/drive/MyDrive/il2_mimetic"
OUT2, OUT3, OUT4 = (f"{PROJ}/stage2_backbones", f"{PROJ}/stage3_seqs",
                    f"{PROJ}/stage4_fold")
OUTT = f"{PROJ}/analysis_tm"
os.makedirs(OUTT, exist_ok=True)

subprocess.run(["pip", "-q", "install", "tmtools", "biopython"], check=True)

from tmtools import tm_align
from tmtools.io import get_structure, get_residue_data

def load_ca(path, chain_id=None):
    """CA coordinates + sequence for one chain."""
    s = get_structure(path)
    chains = list(s.get_chains())
    ch = next((c for c in chains if c.id == chain_id), chains[0])
    return get_residue_data(ch)

def tmscore(path_a, path_b, chain_a=None, chain_b=None):
    ca, sa = load_ca(path_a, chain_a)
    cb, sb = load_ca(path_b, chain_b)
    r = tm_align(ca, cb, sa, sb)
    return max(r.tm_norm_chain1, r.tm_norm_chain2)

cand = pd.read_csv(f"{OUT4}/stage5_candidates.tsv", sep="\t")
cand["pdb"] = cand.seq_id.apply(lambda s: f"{OUT4}/pdb/{s}.pdb")
print(f"{len(cand)} candidates, all present: "
      f"{all(os.path.exists(p) for p in cand.pdb)}")

Mounted at /content/drive
12 candidates, all present: True


In [ ]:
# All-vs-all TM-score across the 12 candidates. This matrix is the
# input Stage 5 uses to pick the diverse six, so it is written before
# anything else depends on it
n = len(cand)
M = np.eye(n)
for i, j in itertools.combinations(range(n), 2):
    M[i, j] = M[j, i] = tmscore(cand.pdb.iloc[i], cand.pdb.iloc[j])

tm = pd.DataFrame(M, index=cand.seq_id, columns=cand.seq_id).round(3)
tm.to_csv(f"{OUTT}/candidates_tm_matrix.tsv", sep="\t")

off = M[np.triu_indices(n, 1)]
print(f"pairwise TM-score across {n} candidates")
print(f"  min {off.min():.3f}  median {np.median(off):.3f}  max {off.max():.3f}")
print(f"  pairs > 0.5 : {(off > 0.5).sum()}/{len(off)}")
print(f"  pairs > 0.7 : {(off > 0.7).sum()}/{len(off)}")
print(f"  pairs > 0.8 : {(off > 0.8).sum()}/{len(off)}")

print("\nmost similar pairs:")
pairs = [(cand.seq_id.iloc[i], cand.seq_id.iloc[j], M[i, j])
         for i, j in itertools.combinations(range(n), 2)]
for a, b, v in sorted(pairs, key=lambda x: -x[2])[:6]:
    print(f"  {v:.3f}  {a}  |  {b}")

pairwise TM-score across 12 candidates
  min 0.598  median 0.892  max 0.979
  pairs > 0.5 : 66/66
  pairs > 0.7 : 64/66
  pairs > 0.8 : 47/66

most similar pairs:
  0.979  pad0_len100_37_fixContacts_2  |  pad0_len100_29_fixContacts_1
  0.962  pad2_len100_28_fixContacts_3  |  pad2_len100_7_fixAll_3
  0.957  pad0_len100_0_fixContacts_1  |  pad0_len100_29_fixContacts_1
  0.951  pad0_len100_29_fixContacts_1  |  pad0_len100_13_fixContacts_3
  0.950  pad2_len100_28_fixContacts_3  |  pad2_len100_27_fixContacts_0
  0.949  pad0_len100_0_fixContacts_1  |  pad0_len100_37_fixContacts_2


In [ ]:
# Does structural similarity track the Stage 2 alpha filter? All 152
# backbones compared within survivors, within a size-matched random
# sample of failures, and across the two groups. Note that pairwise
# distances from n structures give n(n-1)/2 non-independent values,
# which the significance test below does not account for
m2 = pd.read_csv(f"{OUT2}/metrics.tsv", sep="\t")
print(f"{len(m2)} backbones, {m2['pass'].sum()} cleared α")

# load every CA set once
cache = {}
for _, r in m2.iterrows():
    p = f"{OUT2}/{r.variant}/{r.design}.pdb"
    if os.path.exists(p):
        cache[r.design] = load_ca(p)
print(f"loaded {len(cache)} structures")

def tm_cached(a, b):
    ca, sa = cache[a]
    cb, sb = cache[b]
    r = tm_align(ca, cb, sa, sb)
    return max(r.tm_norm_chain1, r.tm_norm_chain2)

surv = [d for d in m2.loc[m2["pass"], "design"] if d in cache]
fail = [d for d in m2.loc[~m2["pass"], "design"] if d in cache]
print(f"survivors {len(surv)}, failures {len(fail)}")

import random
random.seed(0)
fail_s = random.sample(fail, min(29, len(fail)))   # match the survivor count

def dist(group):
    return np.array([tm_cached(a, b)
                     for a, b in itertools.combinations(group, 2)])

d_surv  = dist(surv)
d_fail  = dist(fail_s)
d_cross = np.array([tm_cached(a, b) for a in surv for b in fail_s])

print("\npairwise TM-score distributions")
for label, d in [("within survivors", d_surv),
                 ("within failures ", d_fail),
                 ("across groups   ", d_cross)]:
    print(f"  {label}  n={len(d):5d}  median {np.median(d):.3f}  "
          f"IQR {np.percentile(d,25):.3f}-{np.percentile(d,75):.3f}  "
          f">0.5 {100*(d>0.5).mean():.0f}%")

from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(d_surv, d_fail)
print(f"\nsurvivors vs failures, Mann-Whitney p = {p:.2g}")

np.save(f"{OUTT}/tm_within_survivors.npy", d_surv)
np.save(f"{OUTT}/tm_within_failures.npy", d_fail)

152 backbones, 29 cleared α
loaded 152 structures
survivors 29, failures 123

pairwise TM-score distributions
  within survivors  n=  406  median 0.821  IQR 0.786-0.893  >0.5 100%
  within failures   n=  406  median 0.813  IQR 0.741-0.882  >0.5 100%
  across groups     n=  841  median 0.820  IQR 0.759-0.874  >0.5 100%

survivors vs failures, Mann-Whitney p = 0.0007


In [ ]:
# External scale: each candidate against Neo-2/15 and native IL-2,
# with the Neo-IL-2 score as the reference point for two molecules
# that bind the same receptors by independent architectures
neo = f"{OUT3}/benchmark/6DG6.pdb"
il2 = f"{PROJ}/2B5I.pdb"
with open(f"{PROJ}/stage1_motif/params.json") as fh:
    CHAIN = json.load(fh)["chains"]

rows = []
for _, r in cand.iterrows():
    rows.append({
        "seq_id": r.seq_id,
        "tm_vs_neo": round(tmscore(r.pdb, neo, None, "A"), 3),
        "tm_vs_il2": round(tmscore(r.pdb, il2, None, CHAIN["IL2"]), 3),
    })
ref = pd.DataFrame(rows)
ref.to_csv(f"{OUTT}/candidates_vs_references.tsv", sep="\t", index=False)

print(ref.to_string(index=False))
print("\nvs Neo-2/15:", ref.tm_vs_neo.describe().round(3).to_dict())
print("vs IL-2    :", ref.tm_vs_il2.describe().round(3).to_dict())
print(f"\nNeo-2/15 vs IL-2 (reference point): "
      f"{tmscore(neo, il2, 'A', CHAIN['IL2']):.3f}")

                      seq_id  tm_vs_neo  tm_vs_il2
 pad0_len100_0_fixContacts_1      0.639      0.668
     pad0_len100_15_fixAll_4      0.781      0.516
pad2_len100_28_fixContacts_3      0.699      0.583
     pad0_len100_14_fixAll_1      0.671      0.479
 pad0_len100_2_fixContacts_3      0.600      0.565
      pad2_len100_7_fixAll_3      0.684      0.555
pad2_len100_27_fixContacts_0      0.693      0.520
     pad0_len100_22_fixAll_0      0.651      0.595
pad0_len100_37_fixContacts_2      0.686      0.588
pad0_len100_18_fixContacts_2      0.666      0.589
pad0_len100_29_fixContacts_1      0.798      0.483
pad0_len100_13_fixContacts_3      0.669      0.485

vs Neo-2/15: {'count': 12.0, 'mean': 0.686, 'std': 0.055, 'min': 0.6, '25%': 0.662, '50%': 0.678, '75%': 0.694, 'max': 0.798}
vs IL-2    : {'count': 12.0, 'mean': 0.552, 'std': 0.057, 'min': 0.479, '25%': 0.508, '50%': 0.56, '75%': 0.588, 'max': 0.668}

Neo-2/15 vs IL-2 (reference point): 0.602


In [ ]:
# Pairwise sequence identity, and the baseline that the frozen motif
# forces. The baseline printed here uses the first candidate's
# n_fixed. The panel mixes both arms, so the true floor differs per
# pair
from Bio.Align import PairwiseAligner, substitution_matrices

aln = PairwiseAligner()
aln.mode = "global"
aln.substitution_matrix = substitution_matrices.load("BLOSUM62")
aln.open_gap_score, aln.extend_gap_score = -11, -1

seqs = pd.read_csv(f"{OUT3}/sequences.tsv", sep="\t")
cs = cand.merge(seqs[["seq_id", "sequence"]], on="seq_id")

def pct_id(a, b):
    al = aln.align(a, b)[0]
    s1, s2 = str(al[0]), str(al[1])
    m = sum(x == y and x != "-" for x, y in zip(s1, s2))
    return 100 * m / min(len(a), len(b))

n = len(cs)
S = np.eye(n) * 100
for i, j in itertools.combinations(range(n), 2):
    S[i, j] = S[j, i] = pct_id(cs.sequence.iloc[i], cs.sequence.iloc[j])

off_s = S[np.triu_indices(n, 1)]
off_t = M[np.triu_indices(n, 1)]          # from the TM cell

print(f"pairwise sequence identity across {n} candidates")
print(f"  min {off_s.min():.1f}%  median {np.median(off_s):.1f}%  "
      f"max {off_s.max():.1f}%")
print(f"\nTM-score    median {np.median(off_t):.3f}")
print(f"correlation TM vs identity: {np.corrcoef(off_t, off_s)[0,1]:+.3f}")

# how much is forced by the frozen motif?
n_fixed = int(cs.iloc[0].get("n_fixed", 21))
print(f"\nbaseline identity from the frozen motif alone: "
      f"~{100*n_fixed/cs.length.median():.0f}%")

pd.DataFrame(S, index=cs.seq_id, columns=cs.seq_id).round(1).to_csv(
    f"{OUTT}/candidates_seqid_matrix.tsv", sep="\t")

pairwise sequence identity across 12 candidates
  min 42.9%  median 53.6%  max 69.5%

TM-score    median 0.892
correlation TM vs identity: +0.496

baseline identity from the frozen motif alone: ~20%


#### The results
1. The sequences are not duplicates
| | pairwise sequence identity |
|---|---|
| minimum | 42.9% |
| median | 53.6% |
| maximum | 69.5% |

2. There is only one fold from structural perspective
| | pairwise TM-score |
|---|---|
| minimum | 0.598 |
| median | 0.892 |
| maximum | 0.979 |
| pairs > 0.5 | 66/66 |
| pairs > 0.8 | 47/66 |

TM-score above 0.5 indicates the same fold. Every one of the 66 pairs clears it, and the median of 0.892 indicates near-identical structure rather than merely the same topology. The most similar pair, `37_fixContacts_2` and `29_fixContacts_1`, scores 0.979. At the sequence level there are about 6 distinct solutions. At the structure level there is one topology sampled twelve times. The result underlies in the pipeline design: one RFdiffusion model, one arrangement of three motif segments, two length settings, so the pipeline was never asked for alternative topologies, so it did not produce any.

It also explains why the diversity selection in Stage 5 achieved little: picking the six most dissimilar candidates by greedy max-min lowered maximum pairwise TM from 0.979 to 0.921 but left the median essentially unchanged: 0.891 against 0.892.

3. Shared structure can only partly explain shared sequence
Correlation between TM-score and sequence identity across the 66 pairs: +0.50, accounting for about a quarter of the variance. Similar backbones nudge sequences toward each other but do not determine them, most of the remaining spread comes from ProteinMPNN sampling differently on the same backbone rather than from the backbones differing.

4. The designs resemble Neo-2/15 more than Neo-2/15 resembles IL-2
| | TM-score |
|---|---|
| candidates vs Neo-2/15 | 0.686 ± 0.055 (0.600-0.798) |
| candidates vs native IL-2 | 0.552 ± 0.057 (0.479-0.668) |
| Neo-2/15 vs native IL-2 | 0.602 |

The candidates are structurally closer to Neo-2/15 (0.686) than to
IL-2 (0.552), despite being built from IL-2's motif and never having been compared to Neo-2/15 during design. And they are closer to Neo-2/15 than Neo-2/15 is to IL-2 (0.602).

The likely reading is not functional convergence but shared idealisation. RFdiffusion produces regular, idealised helical bundles; Neo-2/15 is a Rosetta-designed idealised version, however, native IL-2 is an evolved and irregular one. The designs resemble the other de novo molecule because both are idealised, not because they solved the same problem the same way. The sequence-level analysis supports this: when the motif masked, the designs resemble the N-terminal domain of apolipoprotein E (z = +3.2) more than they resemble Neo-2/15 (z = +2.2).

5. The α filter is blind to the fold
| | n pairs | median TM | IQR |
|---|---|---|---|
| within survivors | 406 | 0.821 | 0.786-0.893 |
| within failures | 406 | 0.813 | 0.741-0.882 |
| across groups | 841 | 0.820 | 0.759-0.874 |

Within survivors, within failures and across the two groups, median pairwise TM-score is 0.821, 0.813 and 0.820, with heavily overlapping interquartile ranges.

Taken as a null result, survivors and failures are structurally indistinguishable. Clearing the α filter is not a property of the fold — consistent with Stage 2, where scaffold length predicted clearance across the population and failed entirely as a per-design predictor.



#### Conclusions

1. One fold, twelve samples. All 66 pairs exceed the same-fold threshold and the median pairwise TM-score is 0.892. The candidate set is topologically homogeneous, and every downstream conclusion therefore describes one topology rather than a survey of solutions. The cause is the generation rather than the selection: a single RFdiffusion model, one arrangement of the three motif segments, and two length settings. The pipeline was never asked to explore alternative topologies, so it did not produce any.

2. The sequences are neither duplicates nor independent. The most similar pair shares 69.5%, which is far from the >90% that would mean the same design appearing twice. The median pair shares 53.6%, of which roughly 21 points on average is the motif itself: identical residues the method placed in every design rather than sequence design converging on them. That share is not constant across pairs, ranging from 18% to 40% and highest for the six pairs in which both designs froze all 36 positions. The remaining ~30 points is real
convergence — given similar backbones, ProteinMPNN tends to pick similar residues at the free positions too. No clustering was performed, so the number of distinct sequence groups is not quantified here. Running this analysis before Stage 6 is what stopped twelve near-copies being reported as twelve independent designs.

3. Structural similarity explains only a quarter of sequence similarity. TM-score and pairwise identity correlate at +0.50, accounting for 25% of the variance in identity. What the remaining 75% is attributable to cannot be determined from this panel: all twelve candidates come from twelve different backbones, so there are no same-backbone pairs agains which to separate sequence-design variability from genuine backbone differences. Plausible sources include backbone detail that TM-score does not capture, the two design arms, and sampling stochasticity. Distinguishing them would require carrying several sequences per backbone forward, which Stage 4 deliberately did not do.

4. Resemblance to Neo-2/15 is unlikely to be convergence. The designs score closer to Neo-2/15 (0.686) than Neo-2/15 does to native IL-2 (0.602), despite never having been compared to Neo-2/15 during design. The most likely explanation, though not one this notebook tests, is shared idealisation: generative models and Rosetta both produce regular, idealised helical bundles while evolved proteins are irregular. The sequence-level analysis points the same way, where the motif masked out, the designs resemble the
N-terminal domain of apolipoprotein E (19.8%, z = +3.2) slightly more than they resemble Neo-2/15 (18.3%, z = +2.2).

5. The α filter is not detectable in the fold, as measured here. Survivors and failures have overlapping pairwise similarity distributions, with medians of 0.821 and 0.813. The p = 0.0007 returned by a Mann-Whitney test cannot be interpreted: each group's 406 values are all pairwise combinations of 29 structures and so are not independent observations, and a valid test would permute structure labels rather than pair labels. One caveat on the measure itself is the TM-score normalised by the shorter chain partly normalises out length, and length is the variable Stage 2 found does predict clearance (survivors averaging 102residues against 117 for failures). A length-sensitive comparison might separate the groups, where this one does not.

#### Limitations

1. Pairwise values are not independent observations. Both inferential results in this notebook are affected, and each requires a different correction: the survivor/failure comparison needs a permutation test over structure labels rather than pair labels, and the TM-versus-identity correlation needs a Mantel test. Neither was applied, so both p-values are reported without interpretation.

2. Similarity is not independence. Twelve designs from one model, one motif arrangement and two length settings are correlated by construction. This notebook measures how correlated they are: it says nothing about what the reachable design space contains, which would require varying segment order, segment count, model checkpoint or linker length.